<a href="https://colab.research.google.com/github/jeankonan748/breast-cancer-prediction-bigdata/blob/main/ml_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PHASE 1 — INITIALISATION DU PROJET

Importation des bibliothèques

In [1]:
# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Prétraitement
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

# Modèles ML
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Évaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report


Chargement du Dataset

In [2]:
import os

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
with open('/gdrive/My Drive/foo.txt', 'w') as f:
  f.write('Hello Google Drive!')
!cat '/gdrive/My Drive/foo.txt'

In [ ]:
os.listdir('/gdrive/My Drive/Soutenance M2 ML DL')

In [ ]:
os.listdir('/gdrive/My Drive/Soutenance M2 ML DL/dataset')

In [ ]:
data = pd.read_csv('/gdrive/My Drive/Soutenance M2 ML DL/dataset/breast-cancer.csv')
data

In [ ]:
data.head()

**📌 Analyse Exploratoire Rapide**

In [ ]:
print(data.shape)

In [ ]:
print(data.info())

In [ ]:
print(data.describe())

Vérification des classes

In [ ]:
data['diagnosis'].value_counts()

In [ ]:
#Convertir la variable cible :
data['diagnosis'] = data['diagnosis'].map({'B': 0, 'M': 1})

📌Séparation des Variables

In [ ]:
X = data.drop(['diagnosis', 'id'], axis=1)
y = data['diagnosis']

In [ ]:
y.value_counts()

Normalisation

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Split Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# PHASE 2 — CROSS VALIDATION (K-FOLD)

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "SVM": SVC(probability=True),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier()
}

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=kfold, scoring='accuracy')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

⚙️ PHASE 3 — GRIDSEARCH OPTIMIZATION

🔹 Random Forest Optimisé

In [ ]:
param_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(RandomForestClassifier(), param_rf, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)

print("Best RF Parameters:", grid_rf.best_params_)

🔹 SVM Optimisé

In [ ]:
param_svm = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}

grid_svm = GridSearchCV(SVC(probability=True), param_svm, cv=5, scoring='accuracy')
grid_svm.fit(X_train, y_train)

print("Best SVM Parameters:", grid_svm.best_params_)

📊 **PHASE 4 — ÉVALUATION FINALE**


In [ ]:
def evaluate_model(model):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("AUC:", roc_auc_score(y_test, y_prob))

    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
evaluate_model(grid_rf.best_estimator_)

**Évaluer Random Forest Optimisé**

In [ ]:
best_rf = grid_rf.best_estimator_
evaluate_model(best_rf)

Évaluer SVM Optimisé

In [ ]:
best_svm = grid_svm.best_estimator_
evaluate_model(best_svm)

**📈 PHASE 5 — FEATURE IMPORTANCE**

In [ ]:
importances = best_rf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df.head(10))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title("Top 10 Most Important Features")
plt.show()

**🧠 PHASE 6 — EXPLICABILITÉ AVEC SHAP**

📌Importation de SHAP

In [ ]:
import shap

In [ ]:
shap.initjs()

📌 Création de l’Explainer

On utilise le modèle optimisé :

In [ ]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

In [ ]:
print(f"Shape of shap_values: {shap_values.shape}")

In [ ]:
print(f"Shape of X_test_df: {X_test_df.shape}")

In [ ]:
X_test_df = pd.DataFrame(X_test, columns=X.columns)
shap.summary_plot(shap_values[:, :, 1], X_test_df, feature_names=X.columns)

In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_test_df, feature_names=X.columns, plot_type="bar")

In [ ]:
shap.force_plot(
    explainer.expected_value[1],
    shap_values[0, :, 1],
    X_test[0],
    feature_names=X.columns
)

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[1][0],
        base_values=explainer.expected_value[1],
        data=X_test[0],
        feature_names=X.columns
    )
)

**PHASE 7 — DEEP LEARNING (ANN)**

📌 1️⃣ Installation des bibliothèques

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

📌Construction du modèle ANN

Notre dataset possède 30 variables, donc :

Input layer = 30 neurones

In [ ]:
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))

model.add(Dense(1, activation='sigmoid'))

📌Compilation du modèle

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

📌 5️⃣ Early Stopping (éviter l’overfitting)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

📌 6️⃣ Entraînement du modèle

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

📊 7️⃣ Évaluation du modèle

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

📊 8️⃣ Prédictions

In [ ]:
y_pred_dl = model.predict(X_test)
y_pred_dl = (y_pred_dl > 0.5).astype(int)

📊 9️⃣ Métriques

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_dl))
print("Precision:", precision_score(y_test, y_pred_dl))
print("Recall:", recall_score(y_test, y_pred_dl))
print("F1-score:", f1_score(y_test, y_pred_dl))

📊 Matrice de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred_dl)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Deep Learning")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

📈 10️⃣ Courbes d’apprentissage

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")

plt.legend(['train', 'validation'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")

plt.legend(['train', 'validation'])
plt.show()